# RealSense D435 Depth Parameter Tuning Notebook

This notebook is for testing Intel RealSense D435 RGB / Depth / IR parameters on a black background. It is independent from the global/local detection notebooks and does not modify any project detection code.

Use this notebook to inspect camera option ranges, try RGB and Stereo Module settings, enable light depth filtering, visualize depth validity in real time, and query local median depth by mouse click.

In [13]:
import time
from pathlib import Path
import json
import numpy as np
import cv2
import pyrealsense2 as rs

In [14]:
# Change only these dictionaries when tuning. None means: leave that option unchanged.
# Current fixed RGB-D candidate. RGB is set to the global-only candidate validated on 2026-07-03.
#   Global RGB fixed for 50 Hz lighting: exposure=260, gain=96, brightness=0.
#   white_balance=3600 reduces red cast on the silver base for global detection.
#   manual depth exposure=6000, gain=16, laser_power=150, visual_preset=4,
#   spatial+temporal filters enabled, hole_filling disabled.
# power_line_frequency = 1 is for Germany / 50 Hz.

RGB_SETTINGS = {
    "enable_auto_exposure": False,
    "auto_exposure_priority": 0,
    "power_line_frequency": 1,
    "exposure": 260,
    "gain": 96,
    "enable_auto_white_balance": False,
    "white_balance": 3600,
    "brightness": 0,
    "contrast": 45,
    "saturation": 58,
    "sharpness": 35,
    "gamma": 300,
}

# Optional RGB options. Keep values as None unless you explicitly want to test them.
# Unsupported options are skipped safely.
RGB_EXTRA_SETTINGS = {
    "hue": None,
    "backlight_compensation": None,
}

# Recommended test order for black background / bright gear:
# 1) visual_preset=4, laser_power=150  # already tested: gear still has many invalid pixels
# 2) visual_preset=4, laser_power=120  # tested: center valid ratio dropped
# 3) manual depth exposure=6000, visual_preset=4, laser_power=150  # tested: best so far
# 4) manual depth exposure=5000, visual_preset=4, laser_power=150  # tested: similar, not better
# 5) manual depth exposure=6000, visual_preset=3, laser_power=150  # tested: worse
# 6) best hardware settings plus hole filling enabled  # tested: mask looks full but geometry/detail is worse
# 7) final candidate: best hardware settings without hole filling  # current test
DEPTH_SETTINGS = {
    "enable_auto_exposure": False,
    "exposure": 6000,
    "gain": 16,
    "emitter_enabled": 1,
    "emitter_always_on": None,
    "laser_power": 150,
    "visual_preset": 4,
}

# Optional Stereo Module / depth options. Keep values as None unless testing one at a time.
# Extra tuning order for RGB-D combined performance:
# 1) keep all extra depth options None for a clean baseline  # current test
# 2) change one option at a time only after RGB mask + depth mask are both checked
# 3) confidence_threshold / receiver_gain / noise_filtering can be tested later if needed
# depth_units changes the meter conversion scale; leave it None unless you know why you need it.
# Unsupported options are skipped safely.
DEPTH_EXTRA_SETTINGS = {
    "confidence_threshold": None,
    "min_distance": None,
    "max_distance": None,
    "receiver_gain": None,
    "noise_filtering": None,
    "invalidation_bypass": None,
    "inter_cam_sync_mode": None,
    "frames_queue_size": None,
    "depth_units": None,
}

# Filters are intentionally mild so small raised features are not smoothed away.
FILTER_SETTINGS = {
    "enable_decimation": False,
    "enable_spatial": True,
    "enable_temporal": True,
    "enable_hole_filling": False,

    "spatial_filter_magnitude": 1,
    "spatial_smooth_alpha": 0.4,
    "spatial_smooth_delta": 10,

    "temporal_smooth_alpha": 0.4,
    "temporal_smooth_delta": 20,
}

STREAM_SETTINGS = {
    "width": 640,
    "height": 480,
    "fps": 30,
    "enable_ir": True,
}

VIS_SETTINGS = {
    "depth_min_m": 0.15,
    "depth_max_m": 1.20,
    "click_window_size": 11,
}


In [15]:
def set_option_safe(sensor, option, value, name):
    """Set a RealSense option only when supported and value is not None."""
    if value is None:
        print(f"{name}: skipped because value is None")
        return None
    try:
        if not sensor.supports(option):
            print(f"{name}: not supported")
            return None
        sensor.set_option(option, float(value))
        actual = sensor.get_option(option)
        print(f"{name}: set to {actual}")
        return actual
    except Exception as exc:
        print(f"{name}: failed to set {value}: {exc}")
        return None


def get_option_safe(sensor, option, name):
    """Read a RealSense option only when supported."""
    try:
        if not sensor.supports(option):
            return None
        return sensor.get_option(option)
    except Exception as exc:
        print(f"{name}: failed to read: {exc}")
        return None


def set_option_by_name_safe(sensor, rs_module, option_attr, value, label_prefix):
    """Set an option by enum attribute name, skipping unavailable pyrealsense2 options."""
    name = f"{label_prefix} {option_attr}"
    if value is None:
        print(f"{name}: skipped because value is None")
        return None
    if not hasattr(rs_module.option, option_attr):
        print(f"{name}: option enum not available in this pyrealsense2 version")
        return None
    option = getattr(rs_module.option, option_attr)
    return set_option_safe(sensor, option, value, name)

In [16]:
def _option_display_name(option):
    text = str(option).split(".")[-1].replace("_", " ")
    return text.title()


def _iter_rs_options(rs_module):
    # pyrealsense2 versions differ, so discover option enum members from attributes.
    for attr in sorted(dir(rs_module.option)):
        if attr.startswith("_"):
            continue
        try:
            option = getattr(rs_module.option, attr)
        except Exception:
            continue
        if attr in {"name", "value"}:
            continue
        yield attr, option


def print_all_realsense_options():
    pipeline = rs.pipeline()
    config = rs.config()
    w, h, fps = STREAM_SETTINGS["width"], STREAM_SETTINGS["height"], STREAM_SETTINGS["fps"]
    config.enable_stream(rs.stream.color, w, h, rs.format.bgr8, fps)
    config.enable_stream(rs.stream.depth, w, h, rs.format.z16, fps)

    results = []
    profile = None
    try:
        profile = pipeline.start(config)
        time.sleep(1.0)
        device = profile.get_device()

        for sensor in device.query_sensors():
            sensor_name = sensor.get_info(rs.camera_info.name) if sensor.supports(rs.camera_info.name) else "Unknown Sensor"
            print(f"\nSensor: {sensor_name}")
            sensor_record = {"sensor": sensor_name, "options": []}

            for option_attr, option in _iter_rs_options(rs):
                try:
                    if not sensor.supports(option):
                        continue
                    current = sensor.get_option(option)
                    option_range = sensor.get_option_range(option)
                    record = {
                        "name": _option_display_name(option),
                        "option_attr": option_attr,
                        "current": float(current),
                        "min": float(option_range.min),
                        "max": float(option_range.max),
                        "step": float(option_range.step),
                    }
                    sensor_record["options"].append(record)
                    print(
                        f"{record['name']} current={record['current']} "
                        f"min={record['min']} max={record['max']} step={record['step']}"
                    )
                except Exception as exc:
                    print(f"{option_attr}: failed to inspect: {exc}")

            results.append(sensor_record)

        output_path = Path("realsense_options_current.json")
        output_path.write_text(json.dumps(results, indent=2), encoding="utf-8")
        print(f"\nSaved option snapshot to {output_path.resolve()}")
        return results
    finally:
        if profile is not None:
            pipeline.stop()

In [17]:
def apply_rgb_settings(color_sensor, rs):
    print("\nApplying RGB settings")
    set_option_safe(color_sensor, rs.option.power_line_frequency, RGB_SETTINGS["power_line_frequency"], "RGB power_line_frequency")
    set_option_safe(color_sensor, rs.option.enable_auto_exposure, int(RGB_SETTINGS["enable_auto_exposure"]), "RGB enable_auto_exposure")
    set_option_safe(color_sensor, rs.option.auto_exposure_priority, RGB_SETTINGS["auto_exposure_priority"], "RGB auto_exposure_priority")

    # With auto exposure enabled, exposure/gain remain automatic unless explicitly set in RGB_SETTINGS.
    set_option_safe(color_sensor, rs.option.exposure, RGB_SETTINGS["exposure"], "RGB exposure")
    set_option_safe(color_sensor, rs.option.gain, RGB_SETTINGS["gain"], "RGB gain")

    set_option_safe(color_sensor, rs.option.enable_auto_white_balance, int(RGB_SETTINGS["enable_auto_white_balance"]), "RGB enable_auto_white_balance")
    set_option_safe(color_sensor, rs.option.white_balance, RGB_SETTINGS["white_balance"], "RGB white_balance")
    set_option_safe(color_sensor, rs.option.brightness, RGB_SETTINGS["brightness"], "RGB brightness")
    set_option_safe(color_sensor, rs.option.contrast, RGB_SETTINGS["contrast"], "RGB contrast")
    set_option_safe(color_sensor, rs.option.saturation, RGB_SETTINGS["saturation"], "RGB saturation")
    set_option_safe(color_sensor, rs.option.sharpness, RGB_SETTINGS["sharpness"], "RGB sharpness")
    set_option_safe(color_sensor, rs.option.gamma, RGB_SETTINGS["gamma"], "RGB gamma")

    for option_attr, value in RGB_EXTRA_SETTINGS.items():
        set_option_by_name_safe(color_sensor, rs, option_attr, value, "RGB extra")

In [18]:
def apply_depth_settings(depth_sensor, rs):
    print("\nApplying Depth / Stereo Module settings")
    set_option_safe(depth_sensor, rs.option.enable_auto_exposure, int(DEPTH_SETTINGS["enable_auto_exposure"]), "Depth enable_auto_exposure")

    # With auto exposure enabled, exposure/gain remain automatic unless explicitly set in DEPTH_SETTINGS.
    set_option_safe(depth_sensor, rs.option.exposure, DEPTH_SETTINGS["exposure"], "Depth exposure")
    set_option_safe(depth_sensor, rs.option.gain, DEPTH_SETTINGS["gain"], "Depth gain")
    set_option_safe(depth_sensor, rs.option.emitter_enabled, DEPTH_SETTINGS["emitter_enabled"], "Depth emitter_enabled")
    set_option_safe(depth_sensor, rs.option.emitter_always_on, DEPTH_SETTINGS["emitter_always_on"], "Depth emitter_always_on")
    set_option_safe(depth_sensor, rs.option.laser_power, DEPTH_SETTINGS["laser_power"], "Depth laser_power")
    set_option_safe(depth_sensor, rs.option.visual_preset, DEPTH_SETTINGS["visual_preset"], "Depth visual_preset")

    for option_attr, value in DEPTH_EXTRA_SETTINGS.items():
        set_option_by_name_safe(depth_sensor, rs, option_attr, value, "Depth extra")

In [19]:
def create_depth_filters(rs):
    filters = []

    if FILTER_SETTINGS["enable_decimation"]:
        filters.append(("decimation", rs.decimation_filter()))

    if FILTER_SETTINGS["enable_spatial"]:
        spatial = rs.spatial_filter()
        spatial.set_option(rs.option.filter_magnitude, FILTER_SETTINGS["spatial_filter_magnitude"])
        spatial.set_option(rs.option.filter_smooth_alpha, FILTER_SETTINGS["spatial_smooth_alpha"])
        spatial.set_option(rs.option.filter_smooth_delta, FILTER_SETTINGS["spatial_smooth_delta"])
        filters.append(("spatial", spatial))

    if FILTER_SETTINGS["enable_temporal"]:
        temporal = rs.temporal_filter()
        temporal.set_option(rs.option.filter_smooth_alpha, FILTER_SETTINGS["temporal_smooth_alpha"])
        temporal.set_option(rs.option.filter_smooth_delta, FILTER_SETTINGS["temporal_smooth_delta"])
        filters.append(("temporal", temporal))

    if FILTER_SETTINGS["enable_hole_filling"]:
        filters.append(("hole_filling", rs.hole_filling_filter()))

    print("Depth filters:", ", ".join(name for name, _ in filters) if filters else "disabled")
    return filters


def apply_depth_filters(depth_frame, filters):
    filtered = depth_frame
    for _, filter_obj in filters:
        filtered = filter_obj.process(filtered)
    return filtered

In [20]:
def depth_frame_to_meters(depth_frame, depth_scale):
    depth_raw = np.asanyarray(depth_frame.get_data()).astype(np.float32)
    return depth_raw * float(depth_scale)


def compute_depth_valid_ratio(depth_m, min_m, max_m):
    valid = (depth_m > min_m) & (depth_m < max_m)
    if valid.size == 0:
        return 0.0
    return float(np.count_nonzero(valid) / valid.size)


def query_median_depth(depth_m, u, v, window_size, min_m, max_m):
    h, w = depth_m.shape[:2]
    half = max(0, int(window_size) // 2)
    x0, x1 = max(0, u - half), min(w, u + half + 1)
    y0, y1 = max(0, v - half), min(h, v + half + 1)

    patch = depth_m[y0:y1, x0:x1]
    valid = patch[(patch > min_m) & (patch < max_m)]
    total_count = int(patch.size)
    valid_count = int(valid.size)
    valid_ratio = float(valid_count / total_count) if total_count else 0.0
    median_depth = float(np.median(valid)) if valid_count else None
    return median_depth, valid_count, total_count, valid_ratio

In [21]:
def make_depth_colormap(depth_m, min_m, max_m):
    valid = (depth_m > min_m) & (depth_m < max_m)
    clipped = np.clip(depth_m, min_m, max_m)
    normalized = ((clipped - min_m) / (max_m - min_m) * 255.0).astype(np.uint8)
    colormap = cv2.applyColorMap(normalized, cv2.COLORMAP_TURBO)
    colormap[~valid] = (0, 0, 0)
    return colormap


def make_valid_mask_view(depth_m, min_m, max_m):
    valid = (depth_m > min_m) & (depth_m < max_m)
    return (valid.astype(np.uint8) * 255)

In [22]:
LATEST_DEPTH_M = None
LAST_CLICK_INFO = None


def on_mouse_click(event, x, y, flags, userdata):
    global LATEST_DEPTH_M, LAST_CLICK_INFO
    if event != cv2.EVENT_LBUTTONDOWN or LATEST_DEPTH_M is None:
        return

    min_m = VIS_SETTINGS["depth_min_m"]
    max_m = VIS_SETTINGS["depth_max_m"]
    window_size = VIS_SETTINGS["click_window_size"]
    median_depth, valid_count, total_count, valid_ratio = query_median_depth(
        LATEST_DEPTH_M, x, y, window_size, min_m, max_m
    )

    LAST_CLICK_INFO = {
        "u": int(x),
        "v": int(y),
        "median_depth_m": median_depth,
        "valid_pixels": valid_count,
        "total_pixels": total_count,
        "window_valid_ratio": valid_ratio,
    }

    depth_text = "None" if median_depth is None else f"{median_depth:.4f} m"
    print(f"Click at ({x}, {y})")
    print(f"median_depth = {depth_text}")
    print(f"valid_pixels = {valid_count} / {total_count}")
    print(f"window_valid_ratio = {valid_ratio * 100:.1f} %")

In [23]:
def _find_color_sensor(device):
    for sensor in device.query_sensors():
        name = sensor.get_info(rs.camera_info.name) if sensor.supports(rs.camera_info.name) else ""
        if "RGB" in name or "Color" in name:
            return sensor
    return None


def _put_lines(image, lines, origin=(10, 24), line_height=22, color=(255, 255, 255)):
    x, y = origin
    for idx, line in enumerate(lines):
        yy = y + idx * line_height
        cv2.putText(image, line, (x + 1, yy + 1), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 2, cv2.LINE_AA)
        cv2.putText(image, line, (x, yy), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 1, cv2.LINE_AA)


def _current_option_snapshot(color_sensor, depth_sensor):
    return {
        "rgb_exposure": get_option_safe(color_sensor, rs.option.exposure, "RGB exposure") if color_sensor else None,
        "rgb_gain": get_option_safe(color_sensor, rs.option.gain, "RGB gain") if color_sensor else None,
        "rgb_white_balance": get_option_safe(color_sensor, rs.option.white_balance, "RGB white_balance") if color_sensor else None,
        "rgb_brightness": get_option_safe(color_sensor, rs.option.brightness, "RGB brightness") if color_sensor else None,
        "rgb_contrast": get_option_safe(color_sensor, rs.option.contrast, "RGB contrast") if color_sensor else None,
        "rgb_saturation": get_option_safe(color_sensor, rs.option.saturation, "RGB saturation") if color_sensor else None,
        "rgb_sharpness": get_option_safe(color_sensor, rs.option.sharpness, "RGB sharpness") if color_sensor else None,
        "rgb_gamma": get_option_safe(color_sensor, rs.option.gamma, "RGB gamma") if color_sensor else None,
        "depth_exposure": get_option_safe(depth_sensor, rs.option.exposure, "Depth exposure") if depth_sensor else None,
        "depth_gain": get_option_safe(depth_sensor, rs.option.gain, "Depth gain") if depth_sensor else None,
        "laser_power": get_option_safe(depth_sensor, rs.option.laser_power, "Depth laser_power") if depth_sensor else None,
        "visual_preset": get_option_safe(depth_sensor, rs.option.visual_preset, "Depth visual_preset") if depth_sensor else None,
        "emitter_enabled": get_option_safe(depth_sensor, rs.option.emitter_enabled, "Depth emitter_enabled") if depth_sensor else None,
    }


def _print_sensor_options_current(sensor, label):
    if sensor is None:
        print(f"{label}: sensor not found")
        return
    sensor_name = sensor.get_info(rs.camera_info.name) if sensor.supports(rs.camera_info.name) else label
    print(f"\nSensor: {sensor_name}")
    for option_attr, option in _iter_rs_options(rs):
        try:
            if sensor.supports(option):
                print(f"{_option_display_name(option)} current={sensor.get_option(option)}")
        except Exception:
            pass


def _print_live_help():
    print("\nKeys:")
    print("  q / ESC : quit")
    print("  s       : save current RGB/depth/IR images and current settings JSON")
    print("  p       : print current camera options")
    print("  f       : toggle filters on/off")
    print("  i       : toggle IR windows on/off")
    print("  h       : print help")


def _save_current_frame(frame_data, color_sensor, depth_sensor, filters_enabled):
    output_dir = Path("realsense_tuning_results")
    output_dir.mkdir(parents=True, exist_ok=True)
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    prefix = output_dir / f"realsense_tuning_{timestamp}"

    cv2.imwrite(str(prefix) + "_rgb.png", frame_data["color_bgr"])
    cv2.imwrite(str(prefix) + "_depth_colormap.png", frame_data["depth_colormap"])
    cv2.imwrite(str(prefix) + "_valid_mask.png", frame_data["valid_mask_view"])
    np.save(str(prefix) + "_depth_m.npy", frame_data["depth_m"])

    if frame_data.get("ir_left") is not None:
        cv2.imwrite(str(prefix) + "_ir_left.png", frame_data["ir_left"])
    if frame_data.get("ir_right") is not None:
        cv2.imwrite(str(prefix) + "_ir_right.png", frame_data["ir_right"])

    settings = {
        "timestamp": timestamp,
        "rgb_settings": RGB_SETTINGS,
        "rgb_extra_settings": RGB_EXTRA_SETTINGS,
        "depth_settings": DEPTH_SETTINGS,
        "depth_extra_settings": DEPTH_EXTRA_SETTINGS,
        "filter_settings": FILTER_SETTINGS,
        "stream_settings": STREAM_SETTINGS,
        "vis_settings": VIS_SETTINGS,
        "filters_enabled_live": filters_enabled,
        "current_options": _current_option_snapshot(color_sensor, depth_sensor),
        "last_click_info": LAST_CLICK_INFO,
    }
    Path(str(prefix) + "_settings.json").write_text(json.dumps(settings, indent=2), encoding="utf-8")
    print(f"Saved current frame and settings with prefix: {prefix}")


def run_realsense_parameter_tuning_live():
    global LATEST_DEPTH_M

    pipeline = rs.pipeline()
    config = rs.config()
    w, h, fps = STREAM_SETTINGS["width"], STREAM_SETTINGS["height"], STREAM_SETTINGS["fps"]
    config.enable_stream(rs.stream.color, w, h, rs.format.bgr8, fps)
    config.enable_stream(rs.stream.depth, w, h, rs.format.z16, fps)
    if STREAM_SETTINGS["enable_ir"]:
        config.enable_stream(rs.stream.infrared, 1, w, h, rs.format.y8, fps)
        config.enable_stream(rs.stream.infrared, 2, w, h, rs.format.y8, fps)

    align = rs.align(rs.stream.color)
    profile = None
    last_frame_data = None
    filters_enabled = True
    show_ir = STREAM_SETTINGS["enable_ir"]
    frame_count = 0
    fps_value = 0.0
    fps_t0 = time.time()

    try:
        profile = pipeline.start(config)
        time.sleep(1.0)
        device = profile.get_device()
        color_sensor = _find_color_sensor(device)
        depth_sensor = profile.get_device().first_depth_sensor()
        depth_scale = depth_sensor.get_depth_scale()

        if color_sensor is not None:
            apply_rgb_settings(color_sensor, rs)
        else:
            print("RGB Camera sensor not found")
        apply_depth_settings(depth_sensor, rs)
        filters = create_depth_filters(rs)

        for window_name in ["RGB", "Depth Colormap", "Depth Valid Mask"]:
            cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
            cv2.setMouseCallback(window_name, on_mouse_click)
        if STREAM_SETTINGS["enable_ir"]:
            cv2.namedWindow("IR Left", cv2.WINDOW_NORMAL)
            cv2.namedWindow("IR Right", cv2.WINDOW_NORMAL)

        _print_live_help()

        while True:
            frames = pipeline.wait_for_frames()
            ir_left_frame = frames.get_infrared_frame(1) if STREAM_SETTINGS["enable_ir"] else None
            ir_right_frame = frames.get_infrared_frame(2) if STREAM_SETTINGS["enable_ir"] else None

            aligned_frames = align.process(frames)
            color_frame = aligned_frames.get_color_frame()
            depth_frame = aligned_frames.get_depth_frame()
            if not color_frame or not depth_frame:
                continue

            if filters_enabled and filters:
                depth_frame = apply_depth_filters(depth_frame, filters)

            color_bgr = np.asanyarray(color_frame.get_data()).copy()
            depth_m = depth_frame_to_meters(depth_frame, depth_scale)
            LATEST_DEPTH_M = depth_m

            min_m = VIS_SETTINGS["depth_min_m"]
            max_m = VIS_SETTINGS["depth_max_m"]
            depth_colormap = make_depth_colormap(depth_m, min_m, max_m)
            valid_mask_view = make_valid_mask_view(depth_m, min_m, max_m)
            valid_ratio = compute_depth_valid_ratio(depth_m, min_m, max_m)

            cy, cx = depth_m.shape[0] // 2, depth_m.shape[1] // 2
            roi = depth_m[max(0, cy - 100):min(depth_m.shape[0], cy + 100), max(0, cx - 100):min(depth_m.shape[1], cx + 100)]
            center_valid_ratio = compute_depth_valid_ratio(roi, min_m, max_m)
            cv2.rectangle(color_bgr, (cx - 100, cy - 100), (cx + 100, cy + 100), (0, 255, 255), 1)
            cv2.rectangle(depth_colormap, (cx - 100, cy - 100), (cx + 100, cy + 100), (0, 255, 255), 1)

            frame_count += 1
            now = time.time()
            if now - fps_t0 >= 0.5:
                fps_value = frame_count / (now - fps_t0)
                frame_count = 0
                fps_t0 = now

            current_options = _current_option_snapshot(color_sensor, depth_sensor)
            filter_text = "on: " + ",".join(name for name, _ in filters) if filters_enabled and filters else "off"
            lines = [
                f"FPS {fps_value:.1f}",
                f"valid {valid_ratio * 100:.1f}% center {center_valid_ratio * 100:.1f}%",
                f"RGB exp {current_options['rgb_exposure']} gain {current_options['rgb_gain']} wb {current_options['rgb_white_balance']}",
                f"D exp {current_options['depth_exposure']} gain {current_options['depth_gain']}",
                f"laser {current_options['laser_power']} preset {current_options['visual_preset']} emitter {current_options['emitter_enabled']}",
                f"filters {filter_text}",
            ]
            if LAST_CLICK_INFO is not None:
                md = LAST_CLICK_INFO["median_depth_m"]
                md_text = "None" if md is None else f"{md:.4f}m"
                lines.append(f"click ({LAST_CLICK_INFO['u']},{LAST_CLICK_INFO['v']}) median {md_text}")
                for img in [color_bgr, depth_colormap]:
                    cv2.drawMarker(img, (LAST_CLICK_INFO["u"], LAST_CLICK_INFO["v"]), (0, 255, 0), cv2.MARKER_CROSS, 18, 2)

            _put_lines(color_bgr, lines)
            _put_lines(depth_colormap, lines)

            ir_left = np.asanyarray(ir_left_frame.get_data()).copy() if ir_left_frame else None
            ir_right = np.asanyarray(ir_right_frame.get_data()).copy() if ir_right_frame else None

            cv2.imshow("RGB", color_bgr)
            cv2.imshow("Depth Colormap", depth_colormap)
            cv2.imshow("Depth Valid Mask", valid_mask_view)
            if STREAM_SETTINGS["enable_ir"] and show_ir:
                if ir_left is not None:
                    cv2.imshow("IR Left", ir_left)
                if ir_right is not None:
                    cv2.imshow("IR Right", ir_right)
            last_frame_data = {
                "color_bgr": color_bgr,
                "depth_m": depth_m,
                "depth_colormap": depth_colormap,
                "valid_mask_view": valid_mask_view,
                "ir_left": ir_left,
                "ir_right": ir_right,
            }

            key = cv2.waitKey(1) & 0xFF
            if key in [ord("q"), 27]:
                break
            if key == ord("s") and last_frame_data is not None:
                _save_current_frame(last_frame_data, color_sensor, depth_sensor, filters_enabled)
            elif key == ord("p"):
                print("\nCurrent camera option summary:")
                print(json.dumps(_current_option_snapshot(color_sensor, depth_sensor), indent=2))
                _print_sensor_options_current(color_sensor, "RGB Camera")
                _print_sensor_options_current(depth_sensor, "Stereo Module")
            elif key == ord("f"):
                filters_enabled = not filters_enabled
                print(f"Filters enabled: {filters_enabled}")
            elif key == ord("i") and STREAM_SETTINGS["enable_ir"]:
                show_ir = not show_ir
                if show_ir:
                    cv2.namedWindow("IR Left", cv2.WINDOW_NORMAL)
                    cv2.namedWindow("IR Right", cv2.WINDOW_NORMAL)
                else:
                    cv2.destroyWindow("IR Left")
                    cv2.destroyWindow("IR Right")
                print(f"IR windows visible: {show_ir}")
            elif key == ord("h"):
                _print_live_help()

    finally:
        if profile is not None:
            pipeline.stop()
        cv2.destroyAllWindows()
        LATEST_DEPTH_M = None
        print("RealSense live tuning stopped.")

In [24]:
run_realsense_parameter_tuning_live()


Applying RGB settings
RGB power_line_frequency: set to 1.0
RGB enable_auto_exposure: set to 0.0
RGB auto_exposure_priority: set to 0.0
RGB exposure: set to 260.0
RGB gain: set to 96.0
RGB enable_auto_white_balance: set to 0.0
RGB white_balance: set to 3600.0
RGB brightness: set to 0.0
RGB contrast: set to 45.0
RGB saturation: set to 58.0
RGB sharpness: set to 35.0
RGB gamma: set to 300.0
RGB extra hue: skipped because value is None
RGB extra backlight_compensation: skipped because value is None

Applying Depth / Stereo Module settings
Depth enable_auto_exposure: set to 0.0
Depth exposure: set to 6000.0
Depth gain: set to 16.0
Depth emitter_enabled: set to 1.0
Depth emitter_always_on: skipped because value is None
Depth laser_power: set to 150.0
Depth visual_preset: set to 4.0
Depth extra confidence_threshold: skipped because value is None
Depth extra min_distance: skipped because value is None
Depth extra max_distance: skipped because value is None
Depth extra receiver_gain: skipped b

## Global RGB fixed-parameter capture/evaluation log (2026-07-03)

Purpose: validate whether the fixed RGB settings from this tuning notebook can stabilize `global_rgb_part_detection.ipynb` without modifying the global notebook.

Capture output:

- `realsense_tuning_results/global_rgb_fixed_20260703_122019/`
- 5 RGB frames, 640 x 480, 30 fps, BGR8
- Settings snapshot: `global_rgb_fixed_settings.json`

Fixed RGB settings used:

```python
RGB_SETTINGS = {
    "enable_auto_exposure": False,
    "auto_exposure_priority": 0,
    "power_line_frequency": 1,
    "exposure": 200,
    "gain": 96,
    "enable_auto_white_balance": False,
    "white_balance": 4600,
    "brightness": 0,
    "contrast": 45,
    "saturation": 58,
    "sharpness": 35,
    "gamma": 300,
}
```

All requested RGB options were applied successfully on the RealSense RGB Camera.

Offline evaluation with unchanged `global_rgb_part_detection.ipynb` logic:

- Evaluation output: `realsense_tuning_results/global_rgb_fixed_20260703_122019/offline_eval_from_notebook/`
- Summary: `offline_eval_summary.json`
- Each of the 5 frames produced 5 detections with class list: `Gear_large`, `Screw`, `Gear_large`, `Gear_large`, `Gear_compound`
- Base and base holes were not detected in all 5 frames. Debug masks show the `Silver/base mask` is empty.

Interpretation:

- The fixed RGB settings are temporally stable, but this global scene is underexposed for the current silver/base and white-shaft thresholds.
- Because the current global notebook is RGB-only, depth quality does not affect the global mask detections. Depth settings only matter if the global workflow is later changed to RGB-D.
- The RGB values currently match `local_rgbd_grasp_refinement.ipynb`; keeping this alignment is useful, but global likely needs a brighter fixed RGB variant or adjusted RGB thresholds before migration into `global_rgb_part_detection.ipynb`.



## Global RGB brighter fixed-parameter trials (2026-07-03)

Goal: keep global/local parameter switching small under the staged exclusive-camera ROS workflow, but test a brighter global-only fixed RGB exposure. Depth settings stay aligned with local when depth is enabled; current global mask evaluation remains RGB-only.

Common fixed values:

```python
gain = 96
white_balance = 4600
brightness = 0
contrast = 45
saturation = 58
sharpness = 35
gamma = 300
auto_exposure = False
auto_white_balance = False
power_line_frequency = 1
```

Trial A: `exposure=260`

- Capture: `realsense_tuning_results/global_rgb_exp260_20260703_124129/`
- Offline evaluation: `realsense_tuning_results/global_rgb_exp260_20260703_124129/offline_eval_from_notebook/`
- Result: 5/5 frames still failed to detect `Base` and base holes. Shaft detection partially recovered, but class/count stability was worse than exposure 200.

Trial B: `exposure=320`

- Capture: `realsense_tuning_results/global_rgb_exp320_20260703_124308/`
- Offline evaluation: `realsense_tuning_results/global_rgb_exp320_20260703_124308/offline_eval_from_notebook/`
- Result: 5/5 frames still failed to detect `Base` and base holes. Both shaft detections mostly recovered, but red/gear classes became unstable and base-colored pixels were increasingly included in the red mask.

Interpretation:

- Raising RGB exposure from 200 to 260/320 improves white shaft visibility but does not restore the current global notebook's silver/base mask.
- The failure mode is not depth-related. The current global detection path is RGB-only, so depth quality does not affect these mask results.
- A global-only brighter RGB setting may still be needed, but exposure alone is not enough under the current RGB thresholds. Next candidate should either test a slightly different RGB brightness/contrast combination or evaluate whether the global silver/base thresholds need adjustment before migrating fixed settings into `global_rgb_part_detection.ipynb`.



## Global RGB color-balance tuning trials (2026-07-03)

Goal: continue global-only fixed RGB tuning without modifying `global_rgb_part_detection.ipynb`. The staged ROS plan is still: global and local run in separate phases with exclusive camera access; global applies its complete RGB settings, warms up, detects once, exits; local then applies its RGB-D settings.

Important finding from the previous exposure-only trials: exposure alone did not restore the silver/base mask. The main failure mode was red color cast on the silver base, causing base pixels to leak into the global red mask and then be excluded from the silver/base mask.

Trials run in this round:

1. `exposure=260, white_balance=4600, saturation=40, gamma=300, brightness=0, contrast=45`
   - Capture: `realsense_tuning_results/global_rgb_exp260_sat40_gamma300_br0_ct45_20260703_125114/`
   - Result: Base recovered, but red false positives increased strongly; only 2 base holes were assigned. Not recommended.

2. `exposure=240, white_balance=4600, saturation=45, gamma=300, brightness=0, contrast=45`
   - Capture: `realsense_tuning_results/global_rgb_exp240_sat45_gamma300_br0_ct45_20260703_125313/`
   - Result: Unstable. Base was detected only 1/3 frames and only 1 unknown hole was assigned in that frame. Not recommended.

3. `exposure=260, white_balance=3600, saturation=58, gamma=300, brightness=0, contrast=45`
   - Capture: `realsense_tuning_results/global_rgb_exp260_wb3600_sat58_gamma300_br0_ct45_20260703_125425/`
   - Result: 3/3 frames detected Base and all 3 base holes with correct semantic names. Red false positives were removed. This is the best candidate.

4. `exposure=260, white_balance=3600, saturation=58, gamma=350, brightness=0, contrast=45`
   - Capture: `realsense_tuning_results/global_rgb_exp260_wb3600_sat58_gamma350_br0_ct45_20260703_125618/`
   - Result: Base and holes remained stable, but the missing `Shaft_small` was not recovered. No advantage over gamma 300.

5. `exposure=260, white_balance=3600, saturation=58, gamma=300, brightness=8, contrast=45`
   - Capture: `realsense_tuning_results/global_rgb_exp260_wb3600_sat58_gamma300_br8_ct45_20260703_125759/`
   - Result: Base and holes remained stable, but `Shaft_small` was not recovered and base center shifted slightly more. No advantage over brightness 0.

Final 5-frame candidate validation:

```python
GLOBAL_RGB_CANDIDATE = {
    "enable_auto_exposure": False,
    "auto_exposure_priority": 0,
    "power_line_frequency": 1,
    "exposure": 260,
    "gain": 96,
    "enable_auto_white_balance": False,
    "white_balance": 3600,
    "brightness": 0,
    "contrast": 45,
    "saturation": 58,
    "sharpness": 35,
    "gamma": 300,
}
```

- Capture: `realsense_tuning_results/global_rgb_candidate_global_exp260_wb3600_sat58_gamma300_20260703_125923/`
- Offline evaluation: `realsense_tuning_results/global_rgb_candidate_global_exp260_wb3600_sat58_gamma300_20260703_125923/offline_eval_from_notebook/`
- Base: 5/5 detected
- Base holes: 5/5 detected, all 3 holes assigned (`base_hole_large`, `base_hole_small`, `base_hole_screw`)
- Red parts: stable semantic set in all 5 frames (`Gear_small`, `Gear_large`, `Gear_compound`, `Screw`)
- Shaft: `Shaft_large` detected in 4/5 frames; `Shaft_small` not recovered by these camera-parameter changes

Recommendation:

- Use the final candidate as the current global-only fixed RGB camera setting if the priority is stable base/hole/global coarse localization.
- Keep local RGB-D settings unchanged, including local depth settings.
- In ROS integration, make global apply this complete RGB setting set, discard warmup frames, run one detection, then stop/release the camera before local starts.
- Do not keep increasing exposure. The remaining `Shaft_small` issue is likely better handled in global detection thresholds/geometry if that target is required globally.



In [25]:
# BEST_GLOBAL_CAMERA_SETTINGS_RECORD_20260703
#
# Best fixed GLOBAL camera parameters for the staged ROS workflow.
# Intended use:
#   - Global node exclusively owns the RealSense camera.
#   - Apply the complete RGB option set below.
#   - Discard warmup frames before running the single global detection.
#   - Stop/release the camera before the local node starts.
#
# Notes:
#   - Current global detection is RGB-only; depth quality does not affect the
#     global mask detections unless the global pipeline is later changed to RGB-D.
#   - This global RGB setting is brighter than local and uses cooler white balance
#     to reduce red cast on the silver base.
#   - Validated on 2026-07-03 with 5/5 Base detections and 5/5 three base-hole
#     semantic assignments.
#
# GLOBAL_RGB_SETTINGS = {
#     "enable_auto_exposure": False,
#     "auto_exposure_priority": 0,
#     "power_line_frequency": 1,  # Germany / 50 Hz
#     "exposure": 260,
#     "gain": 96,
#     "enable_auto_white_balance": False,
#     "white_balance": 3600,
#     "brightness": 0,
#     "contrast": 45,
#     "saturation": 58,
#     "sharpness": 35,
#     "gamma": 300,
# }
#
# GLOBAL_DEPTH_SETTINGS = {
#     # Global is currently RGB-only. If a global RGB-D stream is enabled later,
#     # keep depth aligned with the local fixed depth settings below to avoid
#     # extra camera-state switching between stages.
#     "enable_auto_exposure": False,
#     "exposure": 6000,
#     "gain": 16,
#     "emitter_enabled": 1,
#     "emitter_always_on": None,
#     "laser_power": 150,
#     "visual_preset": 4,
# }
#
# -----------------------------------------------------------------------------
#
# Best fixed LOCAL camera parameters for local RGB-D grasp refinement.
# Intended use:
#   - Local node exclusively owns the RealSense camera after global exits.
#   - Apply the complete RGB-D option set below.
#   - Align depth to color.
#   - Apply mild spatial + temporal depth filters.
#   - Discard warmup frames before local refinement.
#
# LOCAL_RGB_SETTINGS = {
#     "enable_auto_exposure": False,
#     "auto_exposure_priority": 0,
#     "power_line_frequency": 1,  # Germany / 50 Hz
#     "exposure": 200,
#     "gain": 96,
#     "enable_auto_white_balance": False,
#     "white_balance": 4600,
#     "brightness": 0,
#     "contrast": 45,
#     "saturation": 58,
#     "sharpness": 35,
#     "gamma": 300,
# }
#
# LOCAL_DEPTH_SETTINGS = {
#     "enable_auto_exposure": False,
#     "exposure": 6000,
#     "gain": 16,
#     "emitter_enabled": 1,
#     "emitter_always_on": None,
#     "laser_power": 150,
#     "visual_preset": 4,
# }
#
# LOCAL_DEPTH_FILTER_SETTINGS = {
#     "enable_decimation": False,
#     "enable_spatial": True,
#     "enable_temporal": True,
#     "enable_hole_filling": False,
#     "spatial_filter_magnitude": 1,
#     "spatial_smooth_alpha": 0.4,
#     "spatial_smooth_delta": 10,
#     "temporal_smooth_alpha": 0.4,
#     "temporal_smooth_delta": 20,
# }

